In [ ]:
import pandas as pd
import numpy as np 
import math
from tqdm import tqdm
import os
import sys

import warnings
warnings.filterwarnings('ignore')

In [ ]:
pd.options.display.max_columns = 10000
pd.options.display.max_rows = 10000

Сначала про то, как устроены эксельки. В них есть строки двух типов – 1 и 2 уровня. Если, например, в закупке участвуют три фирмы, то такая закупка отражается в данных следующим образом: одна строка уровня 1, в которой содержится основная информация о закупке, плюс три строки уровня 2, в которых содержится информация о каждой фирме. Важно понимать, что не вся информация из строки уровня 1 содержится в строках уровня 2. Там нет, например, изначальной цены закупки и типа закупки. То есть при загрузке эту информацию надо аккуратно перенести в строки уровня 2, после чего строку уровня 1 можно удалить. 

Строки уровня 1 для закупок с несколькими фирмами однозначно идентифицируются так: в колонке "Уровень" у них стоит значение 1, в колонке "Поставщик" стоит значение "Несколько поставщиков".

На всякий случай я явном виде указал типы колонок через словарь types. Список cols содержит имена столбцов, которые могут понадобится. Код ниже устроен следующим образом: загружаем эксельки кусками по 30 штук, загружать все сразу очень медленно. Цикл бежит по строкам эксельки и делает следующее: 

1) Если в колонке "Уровень" стоит значение 1, а в колонке "Поставщик" стоит что-то отличное от "Несколько поставщиков", то такая строка просто сохраняется – это закупка с одним поставщиков и у нее нет строк уровня 2.

2) Если в колонке "Уровень" стоит значение 1, а в колонке "Поставщик" стоит "Несколько поставщиков", то мы копируем из этой строки нужную информацию 

3) Если в колонке "Уровень" стоит значение 2, то мы вставляем в строку недостающую информацию, скопированную ранее из строки уровня 1. В рамках одной закупки строки уровня 2 идут строго за строками уровня 1, поэтому ошибок быть не должно

4) Если нам попадается строка, неподходящая под условия выше, то имя эксельки с этой строкой и номер сомой строки копируется в список err, а сама экселька не загружается (чтобы посмотреть и на нее и понять, что с ней не так). По-моему, у меня таких не было.

Загруженные эксельки сохраняются в csv-файлы. Всего получается чуть больше 100 csv-файлов


ВАЖНО: из-за особенностей работы СПАРКа, одна и та же закупка может содержаться в нескольких эксельках, поэтому при загрузке надо удалять дубли. 

In [ ]:
os.chdir('/Users/nvarioshkin/Desktop/CB/data_spark.nosync/data_3') #ставим директорию с эксельками

In [ ]:
cols = ['Уровень', 'Заказчик', 'ИНН', 'Стоимость\n(руб.)', 'Реестровый номер', 'Сфера деятельности',
       'Наименование публикации', 'Регион поставки', 'Город поставки',
       'Дата публикации',
       'Дата \nокончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ',
       'Дата начала подачи заявок/Дата начала исполнения контракта / Дата публикации ППГ',
       'Дата  окончания проведения торгов', 'Поставщик', 'ИНН.1', 'Победитель',
       'Статус допуска', 'Стоимость\n(руб.).1', 'Тип торгов', 'Торговая площадка',
       'Источник финансирования', 'Ссылка на источник']

In [ ]:
types = {
    'Уровень' : object, 
    'Заказчик' : str, 
    'ИНН' : str, 
    'Стоимость\n(руб.)' : object, 
    'Реестровый номер' : str, 
    'Сфера деятельности' : str,
    'Наименование публикации' : str,
    'Регион поставки' : str, 
    'Город поставки' : str, 
    'Дата публикации' : str, 
    'Дата \nокончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ' : str, 
    'Дата начала подачи заявок/Дата начала исполнения контракта / Дата публикации ППГ' : str,
    'Дата  окончания проведения торгов' : str, 
    'Поставщик' : str, 
    'ИНН.1' : str,
    'Победитель' : str, 
    'Статус допуска' : str, 
    'Стоимость\n(руб.).1' : object,  
    'Тип торгов' : str,
    'Торговая площадка' : str, 
    'Источник финансирования' : str, 
    'Ссылка на источник' : str,  
}

In [ ]:
numbers = np.arange(start = 0, stop = 3190, step = 30)

err = []
for i in numbers : 
    
    data_raw = pd.DataFrame()
    

    for filename in tqdm(os.listdir()[i:i + 30]) : 
        if filename.endswith('.xlsx') : 
            foo = pd.read_excel(filename, sheet_name = 'Результаты поиска', skiprows = 1, usecols = cols, dtype = types)
            
            for index in foo.index.values :

                
                if (foo.loc[index, 'Уровень'] == 1) & (foo.loc[index, 'Поставщик'] != 'Несколько поставщиков') :
                    pass
                
                elif (foo.loc[index, 'Уровень'] == 1) & (foo.loc[index, 'Поставщик'] == 'Несколько поставщиков'): 
                    value = foo.loc[index, 'Стоимость\n(руб.)']
                    trade = foo.loc[index, 'Тип торгов']
                    place = foo.loc[index, 'Торговая площадка']
                    fin = foo.loc[index, 'Источник финансирования']
                elif foo.loc[index, 'Уровень'] == 2: 
                    foo.loc[index, 'Стоимость\n(руб.)'] = value
                    foo.loc[index, 'Тип торгов'] = trade
                    foo.loc[index, 'Торговая площадка'] = place
                    foo.loc[index, 'Источник финансирования'] = fin
                else : 
                    err.append([filename, index])
                    foo = pd.DataFrame()
                    break 
            
            
            
            data_raw = data_raw.append(foo)
                        
        
    data_raw.to_csv('/Users/nvarioshkin/Desktop/CB/data_spark.nosync/data_2_x/chunk_' + str(i) + '.csv', index = False)


In [ ]:
os.chdir('/Users/nvarioshkin/Desktop/CB/data_spark.nosync/data_2_x') # ставим директорию с csv-файлами

In [ ]:
# Объеденяем csv в csv побольше в угоду скорости загрузки, при этом удаляя дубли

numbers_1 = np.arange(start = 0, stop = 110, step = 10)


for j in numbers_1 : 
    data_raw = pd.DataFrame()
    
    for item in tqdm(os.listdir()[j:j + 10]) : 
        if item.endswith('.csv') : 
            foo = pd.read_csv(item, dtype = types)
            foo.drop_duplicates(inplace = True)
            data_raw = data_raw.append(foo)
            
    data_raw.to_csv('/Users/nvarioshkin/Desktop/CB/data_spark.nosync/data_3/chunk0_' + str(j) + '.csv', index = False)

In [ ]:
os.chdir('/Users/nvarioshkin/Desktop/CB/data_spark.nosync/data_x') # ставим директорию с новыми csv-файлами

In [ ]:
# Наконец, объеденяем все в один файл, удаляя при этом дубли. 

data_raw_main = pd.DataFrame()
for item in tqdm(os.listdir()) : 
    if item.endswith('.csv') : 
        foo = pd.read_csv(item, dtype = types)
        foo.drop_duplicates(inplace = True)
        data_raw_main = data_raw_main.append(foo)
        
data_raw_main.drop_duplicates(inplace = True)
        

## Обработка датасета

In [ ]:
#При загрузке, небольшое количество строк поехало. Их можно удалить следующим образом 

data_raw_main = data_raw_main[(data_raw_main['Уровень'] == '1') | (data_raw_main['Уровень'] == '2')]

In [ ]:
# Как говорилось выше, удаляем строки уровня 1 для закупок с несколькими фирмами 

data_raw_main = data_raw_main[(data_raw_main['Уровень'] != 1) | (data_raw_main['Поставщик'] != 'Несколько поставщиков')]

In [ ]:
# Делаем несколько очевидных преобразований 

In [ ]:
data_raw_main.loc[:, 'Уровень'] = pd.to_numeric(data_raw_main.loc[:, 'Уровень'])
data_raw_main.loc[:, 'Стоимость\n(руб.)'] = pd.to_numeric(data_raw_main.loc[:, 'Стоимость\n(руб.)'])
data_raw_main.loc[:, 'Стоимость\n(руб.).1'] = pd.to_numeric(data_raw_main.loc[:, 'Стоимость\n(руб.).1'])

data_raw_main.loc[(data_raw_main['Сфера деятельности'] == 'Неизвестно'), 'Сфера деятельности'] = np.nan

In [ ]:
foo = (data_raw_main['Ссылка на источник'] == 'Госзакупки 44ФЗ/94ФЗ') | (data_raw_main['Ссылка на источник'] ==  'ПОРТАЛ ПОСТАВЩИКОВ МОСКВЫ ЗМО 44-ФЗ')
data_raw_main.loc[foo, 'Ссылка на источник'] = 44

In [ ]:
foo = (data_raw_main['Ссылка на источник'] == 'Госзакупки 223ФЗ') | (data_raw_main['Ссылка на источник'] ==  'ПОРТАЛ ПОСТАВЩИКОВ МОСКВЫ ЗМО 223-ФЗ')
data_raw_main.loc[foo, 'Ссылка на источник'] = 223

In [ ]:
# Уже после загрузки мне показалось, что некоторые колонки бесполезны, поэтому я их удалил

data_raw_main.drop(['Город поставки', 'Наименование публикации', 'Источник финансирования', 'Торговая площадка'], axis = 1, inplace = True)

In [ ]:
# Среди закупок есть так называемые "Совместные торги", когда у закупки несколько заказчиков. Их оказлось очень неудобно обробатывать, 
# поэтому я решил их удалить вообще (их не очень много). По техническим причинам, я сделал это через groupby

In [ ]:
grouped = data_raw_main.groupby('Реестровый номер')

In [ ]:
share = []
for name, group in tqdm(grouped) : 
    if 'Совместные торги' in group['Заказчик'].values : 
        share.append(name)
        
data_raw_main = data_raw_main[~data_raw_main['Реестровый номер'].isin(share)]

In [ ]:
# В данных нашлась аномалия: некоторые закупки загрузились дважды и не были удалены, потому что с колонке ОКПД у них указано
# несколько кодов, но в разном порядке. Следующий блок кода их удаляет

In [ ]:
cols_1 = ['Уровень', 'Заказчик', 'ИНН', 'Стоимость\n(руб.)', 'Реестровый номер', 'Регион поставки', 'Дата публикации',
       'Дата \nокончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ',
       'Дата начала подачи заявок/Дата начала исполнения контракта / Дата публикации ППГ',
       'Дата  окончания проведения торгов', 'Поставщик', 'ИНН.1', 'Победитель',
       'Статус допуска', 'Стоимость\n(руб.).1', 'Тип торгов',
       'Ссылка на источник']

data_raw_main.drop_duplicates(subset = cols_1, inplace = True)

In [ ]:
# Еще одна аномалия: в некотрых закупках заявки от одной фирмы дублируются. Их не очень много, поэтому все такие закупки я удалил 

In [ ]:
dum = []
for name, group in tqdm(grouped) : 
    if len(group['ИНН.1'].dropna()) != len(set(group['ИНН.1'].dropna())) : 
        dum.append(name)
        
data_raw_main = data_raw_main[~data_raw_main['Реестровый номер'].isin(dum)]

In [ ]:
# Переименуем колонки 

data_raw_main.columns = ['level', 'customer', 'inn_customer', 'price', 'code', 'industry', 'region',
       'start',
       'application_finish',
       'application_start',
       'finish', 'supplier', 'inn_supplier', 'winner',
       'admittance', 'bid', 'type',
       'fz']

# Даты

В данных есть 4 типа дат : 

1. 'start' – дата опубликования закупки

2. 'application_start' – дата начала приема заявок 

3. 'application_finish' – дата окончания приема заявок 

4. 'finish' – дата кончания проведения торгов 

По умолчанию за дату закупки я принимал дату окончания приема заявок. При этом в данных есть закупки, которые начинались в 2015 и заканчивались в 2016, а также те, которые начинались в 2020 и заканчивались в 2021 – такие закупки отнесены к 2016 и 2020 соответственно. 

In [ ]:
data_raw_main.loc[:, 'start'] = pd.to_datetime(data_raw_main.loc[:, 'start'], errors = 'coerce')
data_raw_main.loc[:, 'application_start'] = pd.to_datetime(data_raw_main.loc[:, 'application_start'], errors = 'coerce')
data_raw_main.loc[:, 'application_finish'] = pd.to_datetime(data_raw_main.loc[:, 'application_finish'], errors = 'coerce')
data_raw_main.loc[:, 'finish'] = pd.to_datetime(data_raw_main.loc[:, 'finish'], errors = 'coerce')

data_raw_main['year'] = data_raw_main['application_finish'].dt.year
data_raw_main['month'] = data_raw_main['application_finish'].dt.month
data_raw_main['day'] = data_raw_main['application_finish'].dt.day

In [ ]:
dum = (data_raw_main['year'] == 2021) & (data_raw_main['start'].dt.year == 2020)

data_raw_main.loc[dum, 'year'] = data_raw_main.loc[dum, 'start'].dt.year
data_raw_main.loc[dum, 'month'] = data_raw_main.loc[dum, 'start'].dt.month
data_raw_main.loc[dum, 'day'] = data_raw_main.loc[dum, 'start'].dt.day

In [ ]:
dum = (data_raw_main['year'] == 2015) & (data_raw_main['finish'].dt.year == 2016)

data_raw_main.loc[dum, 'year'] = data_raw_main.loc[dum, 'finish'].dt.year
data_raw_main.loc[dum, 'month'] = data_raw_main.loc[dum, 'finish'].dt.month
data_raw_main.loc[dum, 'day'] = data_raw_main.loc[dum, 'finish'].dt.day

In [ ]:
# При в данных также оказались закупки, которые попали целиком в 2015 или 2021. Их я удалил

data_raw_main = data_raw_main[data_raw_main['year'].isin([2016, 2017, 2018, 2019, 2020])]

In [ ]:
# Типов закупок очень много, поэтому я выделил основные, а остальные отнес в категорию other. На всякий случай, я создал для этого 
# новую колонку

data_raw_main['type_corrected'] = pd.Series()

data_raw_main.loc[data_raw_main['type'] == 'Аукцион электронный', 'type_corrected'] = 'Аукцион электронный'
data_raw_main.loc[data_raw_main['type'] == 'Запрос котировок', 'type_corrected'] = 'Запрос котировок'
data_raw_main.loc[data_raw_main['type'] == 'Запрос котировок в электронной форме', 'type_corrected'] = 'Запрос котировок в электронной форме'
data_raw_main.loc[data_raw_main['type'] == 'Запрос предложений в электронной форме', 'type_corrected'] = 'Запрос предложений в электронной форме'
data_raw_main.loc[data_raw_main['type'] == 'Запрос предложений', 'type_corrected'] = 'Запрос предложений'
data_raw_main.loc[data_raw_main['type'] == 'Запрос цен', 'type_corrected'] = 'Запрос цен'
data_raw_main.loc[data_raw_main['type'] == 'Закупка у единственного поставщика (исполнителя, подрядчика)', 'type_corrected'] = 'Закупка у единственного поставщика (исполнителя, подрядчика)'
data_raw_main.loc[data_raw_main['type'] == 'Конкурс открытый', 'type_corrected'] = 'Конкурс открытый'
data_raw_main.loc[data_raw_main['type_corrected'].isnull(), 'type_corrected'] = 'Other'

In [ ]:
regions = {'Республика Адыгея' : '01',
           'Республика Башкортостан' : '02', 
           'Республика Бурятия' : '03', 
           'Республика Алтай' : '04', 
           'Республика Дагестан' : '05',
           'Республика Ингушетия' : '06', 
           'Кабардино-Балкарская Республика' : '07', 
           'Республика Калмыкия' : '08', 
           'Карачаево-Черкесская Республика' : '09', 
           'Республика Карелия' : '10', 
           'Республика Коми' : '11', 
           'Республика Марий Эл' : '12', 
           'Республика Мордовия' : '13', 
           'Республика Саха' : '14', 
           'Республика Северная Осетия-Алания' : '15',
           'Республика Татарстан' : '16', 
           'Республика Тыва' : '17', 
           'Удмуртская Республика' : '18', 
           'Республика Хакасия' : '19', 
           'Чеченская Республика' : '20', 
           'Чувашская Республика - Чувашия' : '21', 
           'Алтайский край' : '22', 
           'Краснодарский край' : '23', 
           'Красноярский край' : '24', 
           'Приморский край' : '25', 
           'Ставропольский край' : '26', 
           'Хабаровский край' : '27', 
           'Амурская область' : '28', 
           'Архангельская область' : '29', 
           'Астраханская область' : '30', 
           'Белгородская область' : '31', 
           'Брянская область' : '32', 
           'Владимирская область' : '33', 
           'Волгоградская область' : '34', 
           'Вологодская область' : '35',
           'Воронежская область' : '36', 
           'Ивановская область' : '37',
           'Иркутская область' : '38', 
           'Калининградская область' : '39',
           'Калужская область' : '40', 
           'Камчатский край' : '41', 
           'Кемеровская область' : '42', 
           'Кировская область' : '43', 
           'Костромская область' : '44', 
           'Курганская область' : '45', 
           'Курская область' : '46', 
           'Ленинградская область' : '47',
           'Липецкая область' : '48', 
           'Магаданская область' : '49', 
           'Московская область' : '50', 
           'Мурманская область' : '51', 
           'Нижегородская область' : '52', 
           'Новгородская область' : '53',
           'Новосибирская область' : '54', 
           'Омская область' : '55', 
           'Оренбургская область' : '56', 
           'Орловская область' : '57', 
           'Пензенская область' : '58', 
           'Пермский край' : '59', 
           'Псковская область' : '60', 
           'Ростовская область' : '61', 
           'Рязанская область' : '62', 
           'Самарская область' : '63', 
           'Саратовская область' : '64',
           'Сахалинская область' : '65', 
           'Свердловская область' : '66', 
           'Смоленская область' : '67', 
           'Тамбовская область' : '68', 
           'Тверская область' : '69', 
           'Томская область' : '70', 
           'Тульская область' : '71', 
           'Тюменская область' : '72', 
           'Ульяновская область' : '73', 
           'Челябинская область' : '74', 
           'Забайкальский край' : '75', 
           'Ярославская область' : '76', 
           'Москва' : '77', 
           'Санкт-Петербург' : '78', 
           'Еврейская автономная область' : '79', 
           'Ненецкий автономный округ' : '83',
           'Ханты-Мансийский автономный округ - Югра' : '86', 
           'Чукотский автономный округ' : '87', 
           'Ямало-Ненецкий автономный округ' : '89', 
           'Корякский округ, входящий в состав Камчатского края' : '41', 
           'Коми-Пермяцкий округ, входящий в состав Пермского края' : '59',
           'Байконур' : '99', 
           'Севастополь' : '92', 
           'Республика Крым' : '91',
           'Республика Беларусь' : 'Другие страны',
           'Республика Молдова' : 'Другие страны',
           'Республика Узбекистан' : 'Другие страны',
           'Киргизская Республика' : 'Другие страны',
}

In [ ]:
# Кодируем регионы

foo = data_raw_main['region'].copy()
foo = foo.str.split(';').str.get(0)
foo.replace(regions, inplace = True)
data_raw_main['region'] = foo

In [ ]:
# Чистим ОКПД

foo = data_raw_main['industry'].copy()

foo = foo.str.split(']').str.get(0)
foo = foo.str.strip('[')

foo[foo.str.slice(stop = 6) == 'ОКВЭД2'] = np.nan

foo = foo.str.slice(start = 5)
foo = foo.str.strip()

data_raw_main['industry'] = foo

In [ ]:
data_raw_main.to_csv('/Users/nvarioshkin/Desktop/CB/data_spark.nosync/data_3/data_main.csv', index = False)

# Dif-in-Dif

In [ ]:
# Подгружаем датасет

types_1 = {
    'level' : float,
    'customer' : str,
    'inn_customer' : str,
    'price' : float,
    'code' : str,
    'industry' : str,
    'region' : str,
    'supplier' : str,
    'inn_supplier' : str,
    'admittance' : str,
    'bid' : float,
    'type' : str,
    'type_corrected' : str,
    'fz' : float,
    'year' : float,
    'month' : float,
    'day' : float 
}

dt = ['start', 'application_finish', 'application_start', 'finish']


data = pd.read_csv('/Users/nvarioshkin/Desktop/CB/data_spark.nosync/data_3/data_main.csv', dtype = types_1, date_parser = dt)

In [ ]:
del data

In [ ]:
data_raw_main = pd.read_csv('/Users/nvarioshkin/Desktop/CB/data_spark.nosync/data_3/data_main.csv', dtype = types_1, date_parser = dt)

In [ ]:
data_raw_main

In [ ]:
# Для dif-in-dif'а нам нужно выделить две группы фирм: 1) Те, которые выигрывила контракт хотя бы раз 2) Те, которые не выигрывали 
# никогда 


winners = data[(data['winner'] == 'Победитель')]['inn_supplier']
losers = set(data['inn_supplier']) - set(winners) 
only_losers = data[~data['inn_supplier'].isin(winners)]
only_winners = data[data['winner'] == 'Победитель']

dum_1 = only_winners[['inn_supplier', 'year']].groupby('inn_supplier').aggregate(['min']).droplevel(0, axis = 1).reset_index()
dum_1.columns = ['inn', 'first_contract_winner']
dum_1 = dum_1[dum_1['first_contract_winner'].isin([2016, 2017, 2018, 2019])]

Колонка first_contract_winner содержит год, в котором фирма первый раз выиграла контракт. Это вспомогательная колонка, на основе которой создается переменная для dif-in-dif

In [ ]:
# Подгружаем датасет с данными по эффективности
eff = pd.read_csv('/Users/nvarioshkin/Desktop/CB/efficiency data.nosync/lp(ForContracts).csv', dtype = {'inn' : 'str'})

In [ ]:
eff

In [ ]:
# Оставляем только фирмы, для которых есть данные по росту производительности 

our_inn = eff[~eff['llprod_outputGROWTH'].isnull()]['inn'].unique()
eff = eff[eff['inn'].isin(our_inn)]

In [ ]:
# Добавляем колокни для dif-in-dif
eff = eff.merge(dum_1, how = 'left', on = 'inn')
eff['first_winner_after'] = (eff['first_contract_winner'] <= eff['year']) * 1
eff.loc[eff['first_contract_winner'].isnull(), ['first_winner_after']] = np.nan
eff['loser'] = np.nan
eff.loc[eff['inn'].isin(losers), ['loser']] = 0
eff['dd'] = eff['first_winner_after'].fillna(eff['loser'])

first_winner_after равняется единице для годов, следующих за годом, в котором фирма первый раз выиграла контракт, это тоже вспомогательная колонка. dd – это непосредственно колонка для переменной-прокси dif-in-dif

Я долго думал, как гонять dif-in-dif на подвыборках, и, чтобы не запутаться, решил сделать анологичную колонку dd для каждой подвыборки

In [ ]:
# ФЗ-44 и ФЗ-223

data_223 = data[data['fz'] == 223]
data_44 = data[data['fz'] == 44]

winners_223 = data_223[(data_223['winner'] == 'Победитель')]['inn_supplier']
winners_44 = data_44[(data_44['winner'] == 'Победитель')]['inn_supplier']

losers_223 = set(data_223['inn_supplier']) - set(winners_223) 
losers_44 = set(data_44['inn_supplier']) - set(winners_44) 



In [ ]:
only_winners_223 = data_223[data_223['inn_supplier'].isin(winners_223)]


dum_1 = only_winners_223[['inn_supplier', 'year']].groupby('inn_supplier').aggregate(['min']).droplevel(0, axis = 1).reset_index()
dum_1.columns = ['inn', 'first_contract_winner_223']
dum_1 = dum_1[dum_1['first_contract_winner_223'].isin([2016, 2017, 2018, 2019])]

eff = eff.merge(dum_1, how = 'left', on = 'inn')

eff['first_winner_after_223'] = (eff['first_contract_winner_223'] <= eff['year']) * 1

In [ ]:
only_winners_44 = data_44[data_44['inn_supplier'].isin(winners_44)]


dum_1 = only_winners_44[['inn_supplier', 'year']].groupby('inn_supplier').aggregate(['min']).droplevel(0, axis = 1).reset_index()
dum_1.columns = ['inn', 'first_contract_winner_44']
dum_1 = dum_1[dum_1['first_contract_winner_44'].isin([2016, 2017, 2018, 2019])]

eff = eff.merge(dum_1, how = 'left', on = 'inn')

eff['first_winner_after_44'] = (eff['first_contract_winner_44'] <= eff['year']) * 1

In [ ]:
eff.loc[eff['first_contract_winner_223'].isnull(), ['first_winner_after_223']] = np.nan
eff.loc[eff['first_contract_winner_44'].isnull(), ['first_winner_after_44']] = np.nan

In [ ]:
eff['loser_223'] = np.nan
eff.loc[eff['inn'].isin(losers_223), ['loser_223']] = 0

In [ ]:
eff['loser_44'] = np.nan
eff.loc[eff['inn'].isin(losers_44), ['loser_44']] = 0

In [ ]:
eff['dd_223'] = eff['first_winner_after_223'].fillna(eff['loser_223'])

In [ ]:
eff['dd_44'] = eff['first_winner_after_44'].fillna(eff['loser_44'])

In [ ]:
auc_223 = 'Аукцион электронный'
quot_223 = 'Запрос котировок'
quot_el_223 = 'Запрос котировок в электронной форме'
offer_223 = 'Запрос предложений в электронной форме'

auc_44 = 'Аукцион электронный'
quot_44 = 'Запрос котировок'
quot_el_44 = 'Запрос котировок в электронной форме'
offer_44 = 'Запрос предложений в электронной форме'

In [ ]:
# ФЗ-223, Электронный аукцион

data_auc_223 = data_223[data_223['type_corrected'] == auc_223]
winners_auc_223 = data_auc_223[(data_auc_223['winner'] == 'Победитель')]['inn_supplier']
losers_auc_223 = set(data_auc_223['inn_supplier']) - set(winners_auc_223) 

only_winners_auc_223 = data_auc_223[data_auc_223['inn_supplier'].isin(winners_auc_223)]


dum_1 = only_winners_auc_223[['inn_supplier', 'year']].groupby('inn_supplier').aggregate(['min']).droplevel(0, axis = 1).reset_index()
dum_1.columns = ['inn', 'first_contract_winner_auc_223']
dum_1 = dum_1[dum_1['first_contract_winner_auc_223'].isin([2016, 2017, 2018, 2019])]

eff = eff.merge(dum_1, how = 'left', on = 'inn')

eff['first_winner_after_auc_223'] = (eff['first_contract_winner_auc_223'] <= eff['year']) * 1

eff.loc[eff['first_contract_winner_auc_223'].isnull(), ['first_winner_after_auc_223']] = np.nan

eff['loser_auc_223'] = np.nan
eff.loc[eff['inn'].isin(losers_auc_223), ['loser_auc_223']] = 0

eff['dd_auc_223'] = eff['first_winner_after_auc_223'].fillna(eff['loser_auc_223'])

In [ ]:
# ФЗ-223, Запрос котировок


data_quot_223 = data_223[data_223['type_corrected'] == quot_223]
winners_quot_223 = data_quot_223[(data_quot_223['winner'] == 'Победитель')]['inn_supplier']
losers_quot_223 = set(data_quot_223['inn_supplier']) - set(winners_quot_223) 

only_winners_quot_223 = data_quot_223[data_quot_223['inn_supplier'].isin(winners_quot_223)]


dum_1 = only_winners_quot_223[['inn_supplier', 'year']].groupby('inn_supplier').aggregate(['min']).droplevel(0, axis = 1).reset_index()
dum_1.columns = ['inn', 'first_contract_winner_quot_223']
dum_1 = dum_1[dum_1['first_contract_winner_quot_223'].isin([2016, 2017, 2018, 2019])]

eff = eff.merge(dum_1, how = 'left', on = 'inn')

eff['first_winner_after_quot_223'] = (eff['first_contract_winner_quot_223'] <= eff['year']) * 1

eff.loc[eff['first_contract_winner_quot_223'].isnull(), ['first_winner_after_quot_223']] = np.nan

eff['loser_quot_223'] = np.nan
eff.loc[eff['inn'].isin(losers_quot_223), ['loser_quot_223']] = 0

eff['dd_quot_223'] = eff['first_winner_after_quot_223'].fillna(eff['loser_quot_223'])

In [ ]:
# ФЗ-223, Запрос котировок в электронной форме


data_quot_el_223 = data_223[data_223['type_corrected'] == quot_el_223]
winners_quot_el_223 = data_quot_el_223[(data_quot_el_223['winner'] == 'Победитель')]['inn_supplier']
losers_quot_el_223 = set(data_quot_el_223['inn_supplier']) - set(winners_quot_el_223) 

only_winners_quot_el_223 = data_quot_el_223[data_quot_el_223['inn_supplier'].isin(winners_quot_el_223)]


dum_1 = only_winners_quot_el_223[['inn_supplier', 'year']].groupby('inn_supplier').aggregate(['min']).droplevel(0, axis = 1).reset_index()
dum_1.columns = ['inn', 'first_contract_winner_quot_el_223']
dum_1 = dum_1[dum_1['first_contract_winner_quot_el_223'].isin([2016, 2017, 2018, 2019])]

eff = eff.merge(dum_1, how = 'left', on = 'inn')

eff['first_winner_after_quot_el_223'] = (eff['first_contract_winner_quot_el_223'] <= eff['year']) * 1

eff.loc[eff['first_contract_winner_quot_el_223'].isnull(), ['first_winner_after_quot_el_223']] = np.nan

eff['loser_quot_el_223'] = np.nan
eff.loc[eff['inn'].isin(losers_quot_el_223), ['loser_quot_el_223']] = 0

eff['dd_quot_el_223'] = eff['first_winner_after_quot_el_223'].fillna(eff['loser_quot_el_223'])

In [ ]:
# ФЗ-223, Запрос предложений

data_offer_223 = data_223[data_223['type_corrected'] == offer_223]
winners_offer_223 = data_offer_223[(data_offer_223['winner'] == 'Победитель')]['inn_supplier']
losers_offer_223 = set(data_offer_223['inn_supplier']) - set(winners_offer_223) 

only_winners_offer_223 = data_offer_223[data_offer_223['inn_supplier'].isin(winners_offer_223)]


dum_1 = only_winners_offer_223[['inn_supplier', 'year']].groupby('inn_supplier').aggregate(['min']).droplevel(0, axis = 1).reset_index()
dum_1.columns = ['inn', 'first_contract_winner_offer_223']
dum_1 = dum_1[dum_1['first_contract_winner_offer_223'].isin([2016, 2017, 2018, 2019])]

eff = eff.merge(dum_1, how = 'left', on = 'inn')

eff['first_winner_after_offer_223'] = (eff['first_contract_winner_offer_223'] <= eff['year']) * 1

eff.loc[eff['first_contract_winner_offer_223'].isnull(), ['first_winner_after_offer_223']] = np.nan

eff['loser_offer_223'] = np.nan
eff.loc[eff['inn'].isin(losers_offer_223), ['loser_offer_223']] = 0

eff['dd_offer_223'] = eff['first_winner_after_offer_223'].fillna(eff['loser_offer_223'])

In [ ]:
# ФЗ-44, Электронный аукцион

data_auc_44 = data_44[data_44['type_corrected'] == auc_44]
winners_auc_44 = data_auc_44[(data_auc_44['winner'] == 'Победитель')]['inn_supplier']
losers_auc_44 = set(data_auc_44['inn_supplier']) - set(winners_auc_44) 

only_winners_auc_44 = data_auc_44[data_auc_44['inn_supplier'].isin(winners_auc_44)]


dum_1 = only_winners_auc_44[['inn_supplier', 'year']].groupby('inn_supplier').aggregate(['min']).droplevel(0, axis = 1).reset_index()
dum_1.columns = ['inn', 'first_contract_winner_auc_44']
dum_1 = dum_1[dum_1['first_contract_winner_auc_44'].isin([2016, 2017, 2018, 2019])]

eff = eff.merge(dum_1, how = 'left', on = 'inn')

eff['first_winner_after_auc_44'] = (eff['first_contract_winner_auc_44'] <= eff['year']) * 1

eff.loc[eff['first_contract_winner_auc_44'].isnull(), ['first_winner_after_auc_44']] = np.nan

eff['loser_auc_44'] = np.nan
eff.loc[eff['inn'].isin(losers_auc_44), ['loser_auc_44']] = 0

eff['dd_auc_44'] = eff['first_winner_after_auc_44'].fillna(eff['loser_auc_44'])

In [ ]:
# ФЗ-44, Запрос котировок

data_quot_44 = data_44[data_44['type_corrected'] == quot_44]
winners_quot_44 = data_quot_44[(data_quot_44['winner'] == 'Победитель')]['inn_supplier']
losers_quot_44 = set(data_quot_44['inn_supplier']) - set(winners_quot_44) 

only_winners_quot_44 = data_quot_44[data_quot_44['inn_supplier'].isin(winners_quot_44)]


dum_1 = only_winners_quot_44[['inn_supplier', 'year']].groupby('inn_supplier').aggregate(['min']).droplevel(0, axis = 1).reset_index()
dum_1.columns = ['inn', 'first_contract_winner_quot_44']
dum_1 = dum_1[dum_1['first_contract_winner_quot_44'].isin([2016, 2017, 2018, 2019])]

eff = eff.merge(dum_1, how = 'left', on = 'inn')

eff['first_winner_after_quot_44'] = (eff['first_contract_winner_quot_44'] <= eff['year']) * 1

eff.loc[eff['first_contract_winner_quot_44'].isnull(), ['first_winner_after_quot_44']] = np.nan

eff['loser_quot_44'] = np.nan
eff.loc[eff['inn'].isin(losers_quot_44), ['loser_quot_44']] = 0

eff['dd_quot_44'] = eff['first_winner_after_quot_44'].fillna(eff['loser_quot_44'])

In [ ]:
# ФЗ-44, Запрос котировок в электронной форме

data_quot_el_44 = data_44[data_44['type_corrected'] == quot_el_44]
winners_quot_el_44 = data_quot_el_44[(data_quot_el_44['winner'] == 'Победитель')]['inn_supplier']
losers_quot_el_44 = set(data_quot_el_44['inn_supplier']) - set(winners_quot_el_44) 

only_winners_quot_el_44 = data_quot_el_44[data_quot_el_44['inn_supplier'].isin(winners_quot_el_44)]


dum_1 = only_winners_quot_el_44[['inn_supplier', 'year']].groupby('inn_supplier').aggregate(['min']).droplevel(0, axis = 1).reset_index()
dum_1.columns = ['inn', 'first_contract_winner_quot_el_44']
dum_1 = dum_1[dum_1['first_contract_winner_quot_el_44'].isin([2016, 2017, 2018, 2019])]

eff = eff.merge(dum_1, how = 'left', on = 'inn')

eff['first_winner_after_quot_el_44'] = (eff['first_contract_winner_quot_el_44'] <= eff['year']) * 1

eff.loc[eff['first_contract_winner_quot_el_44'].isnull(), ['first_winner_after_quot_el_44']] = np.nan

eff['loser_quot_el_44'] = np.nan
eff.loc[eff['inn'].isin(losers_quot_el_44), ['loser_quot_el_44']] = 0

eff['dd_quot_el_44'] = eff['first_winner_after_quot_el_44'].fillna(eff['loser_quot_el_44'])

In [ ]:
# ФЗ-44, Запрос предложений

data_offer_44 = data_44[data_44['type_corrected'] == offer_44]
winners_offer_44 = data_offer_44[(data_offer_44['winner'] == 'Победитель')]['inn_supplier']
losers_offer_44 = set(data_offer_44['inn_supplier']) - set(winners_offer_44) 

only_winners_offer_44 = data_offer_44[data_offer_44['inn_supplier'].isin(winners_offer_44)]


dum_1 = only_winners_offer_44[['inn_supplier', 'year']].groupby('inn_supplier').aggregate(['min']).droplevel(0, axis = 1).reset_index()
dum_1.columns = ['inn', 'first_contract_winner_offer_44']
dum_1 = dum_1[dum_1['first_contract_winner_offer_44'].isin([2016, 2017, 2018, 2019])]

eff = eff.merge(dum_1, how = 'left', on = 'inn')

eff['first_winner_after_offer_44'] = (eff['first_contract_winner_offer_44'] <= eff['year']) * 1

eff.loc[eff['first_contract_winner_offer_44'].isnull(), ['first_winner_after_offer_44']] = np.nan

eff['loser_offer_44'] = np.nan
eff.loc[eff['inn'].isin(losers_offer_44), ['loser_offer_44']] = 0

eff['dd_offer_44'] = eff['first_winner_after_offer_44'].fillna(eff['loser_offer_44'])

Итого, для каждой подвыборки у нас есть своя переменная dd, которую можно использовать в dif-in-dif 

In [ ]:
# Создаем отдельную переменную для разделов ОКВЭДА

eff['okved_razdel'] = pd.Series()

x = eff['okved'].copy()
x = x.str.slice(stop = 2).astype(float)


eff.loc[(1 <= x) & (x <= 3), ['okved_razdel']] = 'A'
eff.loc[(5 <= x) & (x <= 9), ['okved_razdel']] = 'B'
eff.loc[(10 <= x) & (x <= 33), ['okved_razdel']] = 'C'
eff.loc[(35 <= x) & (x <= 35), ['okved_razdel']] = 'D'
eff.loc[(36 <= x) & (x <= 39), ['okved_razdel']] = 'E'
eff.loc[(41 <= x) & (x <= 43), ['okved_razdel']] = 'F'
eff.loc[(45 <= x) & (x <= 47), ['okved_razdel']] = 'G'
eff.loc[(49 <= x) & (x <= 53), ['okved_razdel']] = 'H'
eff.loc[(55 <= x) & (x <= 56), ['okved_razdel']] = 'I'
eff.loc[(58 <= x) & (x <= 63), ['okved_razdel']] = 'J'
eff.loc[(64 <= x) & (x <= 66), ['okved_razdel']] = 'K'
eff.loc[(68 <= x) & (x <= 68), ['okved_razdel']] = 'L'
eff.loc[(69 <= x) & (x <= 75), ['okved_razdel']] = 'M'
eff.loc[(77 <= x) & (x <= 82), ['okved_razdel']] = 'N'
eff.loc[(84 <= x) & (x <= 84), ['okved_razdel']] = 'O'
eff.loc[(85 <= x) & (x <= 85), ['okved_razdel']] = 'P'
eff.loc[(86 <= x) & (x <= 88), ['okved_razdel']] = 'Q'
eff.loc[(90 <= x) & (x <= 93), ['okved_razdel']] = 'R'
eff.loc[(94 <= x) & (x <= 96), ['okved_razdel']] = 'S'
eff.loc[(97 <= x) & (x <= 98), ['okved_razdel']] = 'T'
eff.loc[(99 <= x) & (x <= 99), ['okved_razdel']] = 'U'

In [ ]:
# Сохраняем результат в стату
eff.to_stata('difdif.dta', write_index = False)